# Stand-up Comedy — Text Mining

Análisis end-to-end sobre transcripts de stand-up + metadata de IMDb.
Todo el notebook está parametrizado por **filtros** (comediante, año,
rating, votos) — cámbialos en la sección "Filtros" y vuelve a correr
desde ahí hacia abajo para regenerar todas las visualizaciones y
descripciones sobre el subset elegido.

Para una versión interactiva con widgets ver `dashboard/app.py`
(Streamlit). Toda la lógica está en `analysis/core.py` para que
notebook y dashboard compartan el mismo código.


## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings; warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown
sns.set_theme(style="whitegrid")

import nltk
for pkg, kind in [("stopwords", "corpora"), ("punkt", "tokenizers"), ("punkt_tab", "tokenizers")]:
    try: nltk.data.find(f"{kind}/{pkg}")
    except LookupError: nltk.download(pkg, quiet=True)

from analysis import (
    apply_filters, load_unified, tokenize, top_ngrams, sentiment_compound,
    emotion_profile, extract_topics, catchphrases_by_comedian,
    cluster_comedians, predict_rating,
    narrate_filters, narrate_topics, narrate_emotions,
    narrate_catchphrases, narrate_predictor,
)


## 2. Cargar datos

In [ ]:
df_all = load_unified()
print(f"Shows totales: {len(df_all)} | Comediantes únicos: {df_all['comedian'].nunique()}")
df_all[["title", "comedian", "year", "rating", "votes", "runtime_min", "word_count"]].head()


## 3. Filtros — edita aquí

Los filtros son ortogonales: úsalos sueltos o combinados. Todo lo que
viene después usa la variable `df`.


In [ ]:
# === EDITA AQUÍ ===
COMEDIANS = None             # ej: ["Dave Chappelle", "Bo Burnham"]
YEAR_RANGE = (1990, 2025)
MIN_RATING = 0.0
MIN_VOTES = 0
TOP_N_BY_VOTES = None
# ==================

df = apply_filters(df_all, COMEDIANS, YEAR_RANGE, MIN_RATING, MIN_VOTES, TOP_N_BY_VOTES)
Markdown(narrate_filters(df_all, df))


## 4. EDA básico

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
df["word_count"].hist(bins=30, ax=axes[0, 0]); axes[0, 0].set_title("Palabras por show")
df["rating"].dropna().hist(bins=20, ax=axes[0, 1]); axes[0, 1].set_title("Rating IMDb")
df["year"].dropna().astype(int).value_counts().sort_index().plot.bar(ax=axes[1, 0]); axes[1, 0].set_title("Shows por año")
df["runtime_min"].dropna().hist(bins=20, ax=axes[1, 1]); axes[1, 1].set_title("Runtime (min)")
plt.tight_layout(); plt.show()


## 5. Riqueza léxica y velocidad

In [ ]:
df["tokens"] = df["transcript"].map(tokenize)
df["unique_words"] = df["tokens"].map(lambda t: len(set(t)))
df["ttr"] = df["unique_words"] / df["tokens"].map(len).replace(0, np.nan)
df["words_per_min"] = df["word_count"] / df["runtime_min"]

lex = df.groupby("comedian").agg(
    shows=("title", "count"),
    avg_words=("word_count", "mean"),
    avg_ttr=("ttr", "mean"),
    avg_wpm=("words_per_min", "mean"),
    avg_rating=("rating", "mean"),
).sort_values("avg_ttr", ascending=False).head(15)
lex


## 6. N-gramas más frecuentes

In [ ]:
corpus = df["transcript"].tolist()
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, n, title in zip(axes, [1, 2, 3], ["Unigramas", "Bigramas", "Trigramas"]):
    top_ngrams(corpus, n=n).sort_values().plot.barh(ax=ax)
    ax.set_title(f"Top {title}")
plt.tight_layout(); plt.show()


## 7. WordCloud

In [ ]:
from wordcloud import WordCloud
text_blob = " ".join(" ".join(t) for t in df["tokens"])
wc = WordCloud(width=1200, height=600, background_color="white",
               max_words=200, collocations=False).generate(text_blob)
plt.figure(figsize=(14, 7)); plt.imshow(wc, interpolation="bilinear"); plt.axis("off")
plt.title("WordCloud del subset"); plt.show()


## 8. Sentimiento (VADER)

In [ ]:
df["sentiment"] = df["transcript"].map(sentiment_compound)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df["sentiment"].hist(bins=30, ax=axes[0]); axes[0].set_title("Sentimiento (compound) por show")
axes[0].axvline(0, color="red", linestyle="--")
df.groupby("comedian")["sentiment"].mean().sort_values().tail(20).plot.barh(ax=axes[1])
axes[1].set_title("Sentimiento promedio por comediante (top 20)")
plt.tight_layout(); plt.show()


## 9. Emociones discretas — lexicon NRC

8 emociones de Plutchik: anger, anticipation, disgust, fear, joy,
sadness, surprise, trust. El score es la **proporción** de palabras
del transcript tagueadas con esa emoción.


In [ ]:
emo = emotion_profile(df)
Markdown(narrate_emotions(emo))


In [ ]:
# Radar / barras por comediante
import matplotlib.pyplot as plt
top_coms = df["comedian"].value_counts().head(8).index
emo_avg = emo[emo["comedian"].isin(top_coms)].groupby(["comedian", "emotion"])["score"].mean().unstack()

fig, ax = plt.subplots(figsize=(12, 6))
emo_avg.plot.bar(ax=ax)
ax.set_title("Perfil emocional — top comediantes del subset")
ax.set_ylabel("Score NRC (proporción)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


## 10. Topic Modeling (LDA)

Cada show es una **mezcla** de N tópicos latentes. El gráfico muestra
las palabras más representativas de cada tópico.


In [ ]:
N_TOPICS = 8
topic_res = extract_topics(df, n_topics=N_TOPICS, n_top_words=10)
Markdown(narrate_topics(topic_res, df))


In [ ]:
# Tabla de tópicos
pd.DataFrame({
    f"Tópico {i}": words for i, words in topic_res["topics"]
}).head(10)


In [ ]:
# Distribución de tópicos dominantes
import matplotlib.pyplot as plt
dominant = topic_res["doc_topic"].argmax(axis=1)
labels = topic_res["labels"]
counts = pd.Series(dominant).value_counts().sort_index()
plt.figure(figsize=(10, 4))
plt.bar([labels[i] for i in counts.index], counts.values)
plt.title("Shows por tópico dominante"); plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()


## 11. Catchphrases — n-gramas distintivos por comediante

TF-IDF sobre bigramas/trigramas/4-gramas, agregando por comediante.
Lo que aparece arriba: frases que **este comediante usa mucho y los
demás casi nada**.


In [ ]:
cp = catchphrases_by_comedian(df, ngram_range=(2, 4), top_k=10)
Markdown(narrate_catchphrases(cp, top_n=5))


In [ ]:
# Top 5 comediantes con más shows en el subset
top_coms = df["comedian"].value_counts().head(5).index
fig, axes = plt.subplots(1, len(top_coms), figsize=(5 * len(top_coms), 5), sharey=False)
if len(top_coms) == 1: axes = [axes]
for ax, com in zip(axes, top_coms):
    s = cp.get(com)
    if s is not None and len(s):
        s.sort_values().plot.barh(ax=ax)
        ax.set_title(com[:25]); ax.set_xlabel("TF-IDF")
plt.tight_layout(); plt.show()


## 12. Clustering de comediantes (k-means + UMAP)

Agrupa comediantes por similitud de vocabulario y los proyecta a 2D.
Comediantes cerca = estilos parecidos. Color = cluster.


In [ ]:
clus = cluster_comedians(df, k=5, method="umap")
if len(clus):
    plt.figure(figsize=(11, 7))
    for c in sorted(clus["cluster"].unique()):
        sub = clus[clus["cluster"] == c]
        plt.scatter(sub["x"], sub["y"], s=80 + sub["n_shows"]*30,
                    label=f"Cluster {c} (n={len(sub)})", alpha=0.7)
        for _, r in sub.iterrows():
            plt.annotate(r["comedian"][:18], (r["x"], r["y"]), fontsize=8, alpha=0.8)
    plt.legend(); plt.title("Clustering estilístico (UMAP)")
    plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2"); plt.tight_layout(); plt.show()
else:
    print("Subset muy pequeño para clustering.")


## 13. ¿Qué predice el rating? — Ridge interpretable

Modelo lineal sobre TF-IDF + features numéricas. R² bajo es esperado
con n pequeño y texto como input — el interés está en **qué palabras
mueven la aguja**.


In [ ]:
pred = predict_rating(df, alpha=1.0)
Markdown(narrate_predictor(pred))


In [ ]:
if "error" not in pred:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    pred["top_positive"].sort_values().plot.barh(ax=axes[0], color="green")
    axes[0].set_title("Features que SUBEN el rating")
    pred["top_negative"].sort_values().plot.barh(ax=axes[1], color="red")
    axes[1].set_title("Features que BAJAN el rating")
    plt.tight_layout(); plt.show()


## 14. Cruce con IMDb — correlaciones

Cuáles features de texto correlacionan con el rating.


In [ ]:
sub = df.dropna(subset=["rating", "sentiment", "ttr", "word_count"])
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, label in zip(axes, ["sentiment", "ttr", "word_count"],
                          ["Sentimiento", "TTR", "# palabras"]):
    sns.regplot(data=sub, x=col, y="rating", ax=ax, scatter_kws={"alpha": 0.5})
    r = sub[[col, "rating"]].corr().iloc[0, 1]
    ax.set_title(f"{label} vs rating (r={r:.2f})")
plt.tight_layout(); plt.show()


## Más

Para una versión interactiva con filtros visuales y exportación:

```bash
streamlit run dashboard/app.py
```
